In [1]:
# Import Libraries

import tensorflow as tf
import tensorflow.keras as keras
import numpy as np
from tensorflow.keras import layers
from tensorflow.keras import regularizers
import gc
import golois
import sys

print("Python version :"+ sys.version)
print ("Tensorflow version ", tf.__version__)
print("Keras version ", keras.__version__)
print("GPU disponible:", tf.config.list_physical_devices('GPU'))

Python version :3.9.21 (main, Dec 11 2024, 10:21:40) 
[Clang 14.0.6 ]
Tensorflow version  2.16.2
Keras version  3.9.0
GPU disponible: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [2]:
# Configuration

planes = 31
moves = 361
N = 10000
batch = 128


input_data = np.random.randint(2, size=(N, 19, 19, planes))
input_data = input_data.astype ('float32')

policy = np.random.randint(moves, size=(N,))
policy = keras.utils.to_categorical (policy)

value = np.random.randint(2, size=(N,))
value = value.astype ('float32')

end = np.random.randint(2, size=(N, 19, 19, 2))
end = end.astype ('float32')

groups = np.zeros((N, 19, 19, 1))
groups = groups.astype ('float32')

In [3]:
# Get Validation Data

print ("getValidation", flush = True)
golois.getValidation (input_data, policy, value, end)

getValidation


r.shape = (10000, 19, 19, 31)
nbExamples = 10000
nbPositionsSGF = 102208897
nbPositionsSGF = 102208897
loading validation.data


In [4]:
from tensorflow import keras
from tensorflow.keras import layers, regularizers

# Hyperparamètres
l2_reg = 0.0001
filters = 64
trunk = 128
block_iteration = 5

# 🔧 Bloc Squeeze & Excitation
def se_block(input_tensor, filters, ratio=16):
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])

# 🔁 Bloc MobileNet avec SE
def bottleneck_block(x, expand=filters, squeeze=trunk, l2_reg=l2_reg):
    m = layers.Conv2D(expand, (1,1), kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(x)
    m = layers.Activation('relu')(m)

    m = layers.DepthwiseConv2D((3,3), padding='same', kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(m)
    m = layers.Activation('relu')(m)

    m = layers.Conv2D(squeeze, (1,1), kernel_regularizer=regularizers.l2(l2_reg), use_bias=False)(m)

    # 💡 Ajout du bloc SE ici
    m = se_block(m, squeeze)

    # Connexion résiduelle (si dimensions compatibles)
    return layers.Add()([m, x])

# 🧠 Modèle complet
def get_model():
    input = keras.Input(shape=(19, 19, 31), name='board')
    x = layers.Conv2D(trunk, 1, padding='same', kernel_regularizer=regularizers.l2(l2_reg))(input)
    x = layers.ReLU()(x)

    for i in range(block_iteration):
        x = bottleneck_block(x, filters, trunk)

    # Policy head
    policy_head = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias=False,
                                kernel_regularizer=regularizers.l2(l2_reg))(x)
    policy_head = layers.Flatten()(policy_head)
    policy_head = layers.Activation('softmax', name='policy')(policy_head)

    # Value head
    value_head = layers.GlobalAveragePooling2D()(x)
    value_head = layers.Dense(50, activation='relu', kernel_regularizer=regularizers.l2(l2_reg))(value_head)
    value_head = layers.Dropout(0.2)(value_head)
    value_head = layers.Dense(1, activation='sigmoid', name='value',
                              kernel_regularizer=regularizers.l2(l2_reg))(value_head)

    model = keras.Model(inputs=input, outputs=[policy_head, value_head])
    return model

# 📦 Création du modèle
model = get_model()
model.summary()

2025-03-31 10:06:39.628033: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M4 Pro
2025-03-31 10:06:39.628075: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 48.00 GB
2025-03-31 10:06:39.628084: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 18.00 GB
2025-03-31 10:06:39.628108: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:305] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-03-31 10:06:39.628123: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:271] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


ValueError: Unrecognized keyword arguments passed to DepthwiseConv2D: {'kernel_regularizer': <keras.src.regularizers.regularizers.L2 object at 0x3872f1340>}

In [ ]:
from tensorflow.keras import optimizers, callbacks

# Model Compilation and Train

# Epochs number
epochs = 250

# Scheduler de learning rate (comme dans le papier)
def get_learning_rate(epoch):
    if epoch < 25:
        return 0.01
    elif epoch < 50:
        return 0.005
    elif epoch < 100:
        return 0.0005
    elif epoch < 150:
        return 0.00005
    elif epoch < 200:
        return 0.000005
    else:
        return 0.0000005

# Optimiseur Adam avec taux d’apprentissage initial
# 💡 Optimiseur : SGD avec momentum
#optimizer = optimizers.SGD(learning_rate=get_learning_rate(0), momentum=0.9)
optimizer = optimizers.Adam(learning_rate=get_learning_rate(0))

# Poids des pertes
policy_weight = 1.0
value_weight = 4.0

# Compilation du modèle
model.compile(optimizer=optimizer,
              loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
              loss_weights={'policy': policy_weight, 'value': value_weight},
              metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

# Entraînement avec scheduler
for i in range(1, epochs + 1):

    # MAJ du learning rate manuellement
    lr = get_learning_rate(i)
    keras.backend.set_value(model.optimizer.learning_rate, lr)
    print('epoch ' + str(i)+ ', lr '+str(lr))

    # Mise à jour dynamique du batch
    golois.getBatch(input_data, policy, value, end, groups, i * N)

    history = model.fit(input_data,
                        {'policy': policy, 'value': value},
                        epochs=1,
                        batch_size=batch)

    if i % 5 == 0:
        gc.collect()

    if i % epochs == 0:
        golois.getValidation(input_data, policy, value, end)
        val = model.evaluate(input_data,
                             [policy, value], verbose=0, batch_size=batch)
        print("val =", val)
        model.save('mchettih.h5')

epoch 1, lr 0.01
79/79 [==============================] - 11s 50ms/step - loss: 6.8177 - policy_loss: 3.9844 - value_loss: 0.6913 - policy_categorical_accuracy: 0.1853 - value_mse: 0.1186
epoch 2, lr 0.01
79/79 [==============================] - 4s 47ms/step - loss: 6.2247 - policy_loss: 3.4007 - value_loss: 0.6887 - policy_categorical_accuracy: 0.2670 - value_mse: 0.1192
epoch 3, lr 0.01
79/79 [==============================] - 4s 47ms/step - loss: 6.0991 - policy_loss: 3.2753 - value_loss: 0.6884 - policy_categorical_accuracy: 0.2789 - value_mse: 0.1186
epoch 4, lr 0.01
79/79 [==============================] - 4s 48ms/step - loss: 6.0562 - policy_loss: 3.2357 - value_loss: 0.6872 - policy_categorical_accuracy: 0.2813 - value_mse: 0.1186
epoch 5, lr 0.01
79/79 [==============================] - 4s 47ms/step - loss: 6.0645 - policy_loss: 3.2400 - value_loss: 0.6881 - policy_categorical_accuracy: 0.2774 - value_mse: 0.1191
epoch 6, lr 0.01
79/79 [==============================] - 4s 51m

/usr/local/lib/python3.11/dist-packages/keras/src/engine/training.py:3103: UserWarning: You are saving your model as an HDF5 file via `model.save()`. This file format is considered legacy. We recommend using instead the native Keras format, e.g. `model.save('my_model.keras')`.
  saving_api.save_model(


In [ ]:


# ——————————————————————————————————————————
# 🔧 1. Bloc Squeeze & Excitation
# ——————————————————————————————————————————

def se_block(input_tensor, filters, ratio=16):
    se = layers.GlobalAveragePooling2D()(input_tensor)
    se = layers.Reshape((1, 1, filters))(se)
    se = layers.Dense(filters // ratio, activation='relu', use_bias=False)(se)
    se = layers.Dense(filters, activation='sigmoid', use_bias=False)(se)
    return layers.Multiply()([input_tensor, se])

# ——————————————————————————————————————————
# 🔧 2. Bloc MobileNet avec SE intégré
# ——————————————————————————————————————————

def mobile_block_se(x, expand_filters, project_filters, stride=1, ratio=16, l2_reg=0.0001):
    # Expansion
    m = layers.Conv2D(expand_filters, kernel_size=1, padding='same',
                      use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(x)
    m = layers.BatchNormalization()(m)
    m = layers.Activation('relu')(m)

    # Depthwise Convolution (attention ici : depthwise_regularizer !)
    m = layers.DepthwiseConv2D(kernel_size=3, strides=stride, padding='same',
                               use_bias=False, depthwise_regularizer=regularizers.l2(l2_reg))(m)
    m = layers.BatchNormalization()(m)
    m = layers.Activation('relu')(m)

    # Projection
    m = layers.Conv2D(project_filters, kernel_size=1, padding='same',
                      use_bias=False, kernel_regularizer=regularizers.l2(l2_reg))(m)
    m = layers.BatchNormalization()(m)

    # Squeeze & Excitation
    m = se_block(m, project_filters, ratio)

    # Résidu (si même shape)
    if stride == 1 and x.shape[-1] == project_filters:
        m = layers.Add()([x, m])

    return m

# ——————————————————————————————————————————
# 🧠 3. Modèle SE-MobileNet
# ——————————————————————————————————————————

def build_se_mobilenet(blocks=5, expand=1152, project=192):
    input = keras.Input(shape=(19, 19, 31), name='board')
    x = layers.Conv2D(project, 1, padding='same',
                      kernel_regularizer=regularizers.l2(0.0001))(input)
    x = layers.BatchNormalization()(x)
    x = layers.ReLU()(x)

    for _ in range(blocks):
        x = mobile_block_se(x, expand_filters=expand, project_filters=project)

    # Policy head
    policy = layers.Conv2D(1, 1, activation='relu', padding='same', use_bias=False)(x)
    policy = layers.Flatten()(policy)
    policy = layers.Activation('softmax', name='policy')(policy)

    # Value head
    value = layers.GlobalAveragePooling2D()(x)
    value = layers.Dense(50, activation='relu')(value)
    value = layers.Dense(1, activation='sigmoid', name='value')(value)

    return keras.Model(inputs=input, outputs=[policy, value])

# ——————————————————————————————————————————
# ⚙️ 4. Entraînement du modèle
# ——————————————————————————————————————————

# Hyperparamètres
epochs = 250
batch_size = 32
policy_weight = 1.0
value_weight = 4.0

# Scheduler de learning rate (comme dans le papier)
def get_lr(epoch):
    if epoch < 100:
        return 0.0005
    elif epoch < 150:
        return 0.00005
    elif epoch < 200:
        return 0.000005
    else:
        return 0.0000005

# Création du modèle
model = build_se_mobilenet(blocks=32, expand=1152, project=192)
model.summary()

# Optimiseur
optimizer = Adam(learning_rate=get_lr(0))

# Compilation avec pertes pondérées
model.compile(optimizer=optimizer,
              loss={'policy': 'categorical_crossentropy', 'value': 'binary_crossentropy'},
              loss_weights={'policy': policy_weight, 'value': value_weight},
              metrics={'policy': 'categorical_accuracy', 'value': 'mse'})

# ——————————————————————————————————————————
# 🔁 5. Boucle d'entraînement
# ——————————————————————————————————————————

for epoch in range(1, epochs + 1):
    print(f"\n🔁 Epoch {epoch}")

    # MAJ du learning rate
    lr = get_lr(epoch)
    model.optimizer.learning_rate.assign(lr)
    print(f"→ Learning rate: {lr}")

    # Chargement dynamique du batch (adapté à ton pipeline Go)
    golois.getBatch(input_data, policy, value, end, groups, epoch * N)

    # Fit pour une époque
    model.fit(input_data,
              {'policy': policy, 'value': value},
              epochs=1,
              batch_size=batch_size,
              verbose=1)

    # Garbage collector pour libérer mémoire GPU
    if epoch % 5 == 0:
        gc.collect()

    # Évaluation et sauvegarde
    if epoch % epochs == 0:
        golois.getValidation(input_data, policy, value, end)
        val = model.evaluate(input_data, [policy, value], verbose=0, batch_size=batch_size)
        print(f"✅ Validation: {val}")
        model.save('best_se_mobilenet.h5')

Model: "model_1"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 board (InputLayer)          [(None, 19, 19, 31)]         0         []                            
                                                                                                  
 conv2d_145 (Conv2D)         (None, 19, 19, 192)          6144      ['board[0][0]']               
                                                                                                  
 batch_normalization_210 (B  (None, 19, 19, 192)          768       ['conv2d_145[0][0]']          
 atchNormalization)                                                                               
                                                                                                  
 re_lu_3 (ReLU)              (None, 19, 19, 192)          0         ['batch_normalization_21

NameError: name 'Adam' is not defined